In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torch import nn

import torch.nn as nn
import torch.optim as optim
import torch

from sklearn.preprocessing import StandardScaler
import joblib
import copy
import json

import wandb

In [ ]:
data = pd.read_csv("building_database_training_final.csv")

In [ ]:
# Divide into heating and cooling data to use for heating and cooling demand training respectively
data_heating = data[(data['heat_system_type'] == 'gas_boiler') | (data['heat_system_type'] == 'heat_pump') | (data['heat_system_type'] == 'district_heating')]
data_cooling = data[data['cool_system_type'] == 'dx_airco']

In [ ]:
wandb.init(
    project="final_10_4_runs",  # name of your W&B project
    name="FinalSaved_Cooling_H3_S128_LR0.001",
    config={
        "learning_rate": 0.001,
        "batch_size": 64,
        "epochs": 600,
        "hidden_size": 128,
        "hidden_layers": 3,
        "model_type": "MLP"
    }
)

wandb.run.notes =  """ 
one_hot_columns = ['archetype_mapped_1', 'archetype_mapped_2', 'archetype_mapped_3',
                'archetype_mapped_6', 'archetype_mapped_7'] ## 
numeric_columns = ['idf_num_floors', 'pp_ht_floor', 'pp_orientation',
                'idf_total_floor_area', 'idf_exposed_wall_area', 'idf_compact_ratio', 'idf_adiab_wall_area',
                'r_wall_ins', 'r_roof_ins', 'r_floor_ins', 'u_window', 'shgc', 'inf_rate_q4',
                'idf_ground_floor_area', 'idf_window_area', 'lighting', 'electrical_equipment', 'people']

label_column = ['cooling_dem_m2'] ('heating_dem_m2' for heating demand training)
                """

In [ ]:
# Define input features
one_hot_columns = ['archetype_mapped_1', 'archetype_mapped_2', 'archetype_mapped_3',
                'archetype_mapped_6', 'archetype_mapped_7'] ## 
numeric_columns = ['idf_num_floors', 'pp_ht_floor', 'pp_orientation',
                'idf_total_floor_area', 'idf_exposed_wall_area', 'idf_compact_ratio', 'idf_adiab_wall_area',
                'r_wall_ins', 'r_roof_ins', 'r_floor_ins', 'u_window', 'shgc', 'inf_rate_q4',
                'idf_ground_floor_area', 'idf_window_area', 'lighting', 'electrical_equipment', 'people']

label_column = ['cooling_dem_m2'] #'heating_dem_m2' for heating demand training, 'cooling_dem_m2' for cooling demand training

In [ ]:
class NeuralNet_3_Ln_2DBottleneck(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, hidden_size3, output_size):
        super(NeuralNet_3_Ln_2DBottleneck, self).__init__()

        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.ln1 = nn.LayerNorm(hidden_size1)
        self.activation1 = nn.ReLU()

        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.ln2 = nn.LayerNorm(hidden_size2)
        self.activation2 = nn.ReLU()

        self.fc3 = nn.Linear(hidden_size2, hidden_size3)
        self.ln3 = nn.LayerNorm(hidden_size3)
        self.activation3 = nn.ReLU()

        # 2D bottleneck
        self.embedding_2d = nn.Linear(hidden_size3, 2)

        # Regression output now uses the 2D embedding
        self.output = nn.Linear(2, output_size)

    def forward(self, x, return_embedding=False):
        x = self.fc1(x)
        x = self.ln1(x)
        x = self.activation1(x)

        x = self.fc2(x)
        x = self.ln2(x)
        x = self.activation2(x)

        x = self.fc3(x)
        x = self.ln3(x)
        x = self.activation3(x)

        z = self.embedding_2d(x)   # shape: [batch_size, 2]
        y = self.output(z)         # shape: [batch_size, output_size]

        if return_embedding:
            return y, z

        return y

In [ ]:
# Define a custom class for the BuildingDataset
class BuildingDataset(Dataset):
    def __init__(self, inputs, outputs):
        self.inputs = inputs
        self.outputs = outputs

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.outputs[idx]

In [ ]:
# Input and output tensor for heating demand training
one_hot_features = data_heating[one_hot_columns].values
numeric_features = data_heating[numeric_columns].values

input_scaler=StandardScaler()
numeric_scaled = input_scaler.fit_transform(numeric_features)
input_tensor = torch.cat((torch.tensor(one_hot_features, dtype=torch.float32), torch.tensor(numeric_scaled, dtype=torch.float32)), dim=1)

output_scaler=StandardScaler()
output_scaled = output_scaler.fit_transform(data_heating[label_column])
output_tensor = torch.tensor(output_scaled, dtype=torch.float32)

In [ ]:
# Input and output tensor for cooling demand training
one_hot_features = data_cooling[one_hot_columns].values
numeric_features = data_cooling[numeric_columns].values

input_scaler=StandardScaler()
numeric_scaled = input_scaler.fit_transform(numeric_features)
input_tensor = torch.cat((torch.tensor(one_hot_features, dtype=torch.float32), torch.tensor(numeric_scaled, dtype=torch.float32)), dim=1)

target_df = data_cooling[label_column].copy()
target_df['cooling_dem_m2'] = np.log1p(target_df['cooling_dem_m2'].clip(lower=0))

output_scaler=StandardScaler()
output_scaled = output_scaler.fit_transform(target_df)
output_tensor = torch.tensor(output_scaled, dtype=torch.float32)

In [ ]:
seed=42
torch.manual_seed(seed)

In [ ]:
#Split into training and test dataset
X_train_full, X_test, y_train_full, y_test = train_test_split(input_tensor, output_tensor, test_size=0.2, random_state=seed)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=seed) 

train_dataset = BuildingDataset(X_train, y_train)
val_dataset = BuildingDataset(X_val, y_val)
test_dataset = BuildingDataset(X_test, y_test)

X_train.shape, X_val.shape, X_test.shape

In [ ]:
epochs = 600
batch_size = 64
learning_rate = 0.001

input_size=input_tensor.shape[1]
hidden_size1=128
hidden_size2=128
hidden_size3=128
output_size=output_tensor.shape[1]

model = NeuralNet_3_Ln_2DBottleneck(input_size, hidden_size1, hidden_size2, hidden_size3, output_size)
optimizer = optim.Adam(model.parameters(), learning_rate)
criterion = nn.MSELoss()

In [ ]:
def compute_metrics_once(output_scaler, all_labels, all_preds):
    all_preds_np = all_preds.detach().cpu().numpy()
    all_labels_np = all_labels.detach().cpu().numpy()

    mae_per_output = []
    mse_per_output = []
    mape_per_output = []
    mpe_per_output = []

    all_preds_denorm = output_scaler.inverse_transform(all_preds_np)
    all_labels_denorm = output_scaler.inverse_transform(all_labels_np)
    
    cool_idx = label_column.index('cooling_dem_m2')
    all_preds_denorm[:, cool_idx] = np.expm1(all_preds_denorm[:, cool_idx])
    all_labels_denorm[:, cool_idx] = np.expm1(all_labels_denorm[:, cool_idx])
    eps=0.1

    for i in range(all_preds_np.shape[1]):
        y_true = all_labels_denorm[:, i]
        y_pred = all_preds_denorm[:, i]

        valid = np.abs(y_true) > eps
        abs_errors = np.abs(y_pred[valid] - y_true[valid])
        abs_percentage_errors = np.abs((y_pred[valid] - y_true[valid]) / y_true[valid]) * 100
        percentage_errors = ((y_pred[valid] - y_true[valid]) / y_true[valid]) * 100
        mse_errors = ((y_pred[valid] - y_true[valid]) ** 2)

        mae = np.mean(abs_errors)
        mape = np.mean(abs_percentage_errors)
        mpe = np.mean(percentage_errors)
        mse = np.mean(mse_errors)

        mae_per_output.append(mae)
        mse_per_output.append(mse)
        mape_per_output.append(mape)
        mpe_per_output.append(mpe)

    return mae_per_output, mse_per_output, mape_per_output, mpe_per_output

In [ ]:
# Model training & loading in wandb
train_losses = []
train_loader = DataLoader(train_dataset, batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size, shuffle=False)

wandb.watch(model, log="all", log_freq=30)
input_feature_names = one_hot_columns + numeric_columns

best_mape = float("inf")
best_model_state = copy.deepcopy(model.state_dict())
best_epoch = -1

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for batch_features, batch_labels in train_loader:
        optimizer.zero_grad()
        batch_features = batch_features.detach().clone().requires_grad_(True)
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        wandb.log({
            "grad_norm/fc1": model.fc1.weight.grad.norm().item(),
            "grad_norm/fc2": model.fc2.weight.grad.norm().item(),
            "grad_norm/fc3": model.fc3.weight.grad.norm().item(),
            "grad_norm/embedding_2d": model.embedding_2d.weight.grad.norm().item(),
            "grad_norm/output": model.output.weight.grad.norm().item(),
        })

        input_grad_mean_abs = batch_features.grad.detach().abs().mean(dim=0).cpu().numpy()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    train_losses.append(train_loss)

    model.eval()
    val_loss = 0.0
    val_preds_epoch = []
    val_labels_epoch = []

    with torch.no_grad():
        for batch_features, batch_labels in val_loader:
            outputs = model(batch_features)
            loss = criterion(outputs, batch_labels)

            val_loss += loss.item()
            val_preds_epoch.append(outputs.detach())
            val_labels_epoch.append(batch_labels.detach())

    val_loss /= len(val_loader)

    all_val_preds = torch.cat(val_preds_epoch, dim=0)
    all_val_labels = torch.cat(val_labels_epoch, dim=0)
    mae_val, mse_val, mape_val, mpe_val = compute_metrics_once(output_scaler, all_val_labels, all_val_preds)

    if mape_val[0] < best_mape:
        best_mape = np.mean(mape_val)
        best_model_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch + 1

    if epoch % 30 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}")
        print(f"Epoch {epoch+1}/{epochs}, Validation Loss: {val_loss:.4f}")
        print(f"Val MAE: {mae_val}")
        print(f"Val MSE: {mse_val}")
        print(f"Val MAPE: {mape_val}")
        print(f"Val MPE: {mpe_val}")

    wandb_log_payload = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss
    }
    for i, output_name in enumerate(label_column):
        wandb_log_payload[f"val_mae/{output_name}"] = float(mae_val[i])
        wandb_log_payload[f"val_mse/{output_name}"] = float(mse_val[i])
        wandb_log_payload[f"val_mape/{output_name}"] = float(mape_val[i])
        wandb_log_payload[f"val_mpe/{output_name}"] = float(mpe_val[i])

    wandb.log(wandb_log_payload)

In [ ]:
# (Optional) Load best state models
model.load_state_dict(best_model_state)

In [ ]:
# Check performance on test set
model.eval()
test_loss = 0.0

all_preds = []
all_labels = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        test_loss += loss.item()

        all_preds.append(outputs)
        all_labels.append(batch_labels)

test_loss /= len(test_loader)

all_preds = torch.cat(all_preds, dim=0)
all_labels = torch.cat(all_labels, dim=0)

mae_val, mse_val, mape_val, mpe_val = compute_metrics_once(output_scaler, all_labels, all_preds)

print(f"Test Loss: {test_loss:.4f}")
print("MAE:", mae_val)
print("MSE:", mse_val)
print("MAPE:", mape_val)
print("MPE:", mpe_val)

In [ ]:
# Boxplots for final evaluation metrics
output_labels = ['Heating demand']
all_preds_np = all_preds.detach().cpu().numpy()
all_labels_np = all_labels.detach().cpu().numpy()

all_preds_denorm = output_scaler.inverse_transform(all_preds_np)
all_labels_denorm = output_scaler.inverse_transform(all_labels_np)

# Inverse logarithmic transformation (only in the case of cooling demand training)
all_preds_denorm = np.expm1(all_preds_denorm)
all_labels_denorm = np.expm1(all_labels_denorm)

eps = 0.1  # So that the metrics are calculated only on the non-zero values
for i in range(all_preds_np.shape[1]):
    y_pred = all_preds_denorm[:, i]
    y_true = all_labels_denorm[:, i]

    valid = np.abs(y_true) > eps
    coverage = valid.mean() * 100

    if np.any(valid):
        abs_errors = np.abs(y_pred[valid] - y_true[valid])
        abs_percentage_errors = np.abs((y_pred[valid] - y_true[valid]) / y_true[valid]) * 100
        percentage_errors = ((y_pred[valid] - y_true[valid]) / y_true[valid]) * 100

        mae = np.mean(abs_errors)
        mape = np.mean(abs_percentage_errors)
        mpe = np.mean(percentage_errors)
    else:
        abs_errors = np.array([])
        abs_percentage_errors = np.array([])
        percentage_errors = np.array([])
        mae = np.nan
        mape = np.nan
        mpe = np.nan
        mape_median = np.nan
        mpe_median = np.nan

    print(f"Output: {output_labels[i]}")
    print(f"Coverage (|y_true| >= {eps}): {coverage:.1f}%")
    print(f"MAE (valid only): {mae:.2f}")
    print(f"MAPE (valid only): {mape:.2f}%")
    print(f"MPE (valid only): {mpe:.2f}%")

    if abs_percentage_errors.size > 0:
        plt.figure(figsize=(18, 6))

        plt.subplot(1, 3, 1)
        plt.boxplot(abs_errors, vert=True, patch_artist=True)
        plt.title('Absolute Errors (for MAE)')
        plt.xlabel(f'Output: {output_labels[i]}')
        plt.ylabel('Absolute Error (units)')
        plt.grid(axis='y')

        plt.subplot(1, 3, 2)
        plt.boxplot(abs_percentage_errors, vert=True, patch_artist=True)
        plt.title('Absolute Percentage Errors (for MAPE)')
        plt.xlabel(f'Output: {output_labels[i]}')
        plt.ylabel('Percentage Error (%)')
        plt.grid(axis='y')

        plt.subplot(1, 3, 3)
        plt.boxplot(percentage_errors, vert=True, patch_artist=True)
        plt.title('Percentage Errors (for MPE)')
        plt.xlabel(f'Output: {output_labels[i]}')
        plt.ylabel('Percentage Error (%)')
        plt.grid(axis='y')

        plt.tight_layout()
        plt.show()
    else:
        print('No valid samples for error boxplots at this threshold.')

In [ ]:
# Save model, input and output scalers and metadata
model_path = "OUT_models/Cooling demand/2Dplot/5848/pred_cool_dem_beststate.pth"
input_scaler_path = "OUT_models/Cooling demand/2Dplot/5848/pred_cool_dem_input_scaler.pkl"
output_scaler_path = "OUT_models/Cooling demand/2Dplot/5848/pred_cool_dem_output_scaler.pkl"
meta_path = "OUT_models/Cooling demand/2Dplot/5848/pred_cool_dem_beststate_meta.json"

model_config = {
    "input_size": model.fc1.in_features,
    "hidden_size1": 128,
    "hidden_size2": 128,
    "hidden_size3": 128,
    "embedding_2d": 2,
    "output_size": 1
}

checkpoint = {
    "model_state_dict": model.state_dict(),
    "model_config": model_config
}
torch.save(checkpoint, model_path)

joblib.dump(input_scaler, input_scaler_path)
joblib.dump(output_scaler, output_scaler_path)

meta = {
    "one_hot_columns": one_hot_columns,
    "numeric_columns": numeric_columns,
    "label_columns": 'cooling_dem_m2',
    }

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

In [ ]:
wandb.finish()